# 拆分数据

In [2]:
import pandas as pd
import os
from onekey_algo import get_param_in_cwd

bagsize = get_param_in_cwd('bagsize')
all_features = pd.read_csv(get_param_in_cwd('feature_file'), dtype={'ID': str})
all_features

In [ ]:
order_by_col = get_param_in_cwd('order_by_col', None)

if order_by_col is None:
    sel_features = []
    for sid in set(all_features['ID']):
        sample_features = all_features[all_features['ID'] == sid]
        if sample_features.shape[0] > bagsize:
            sample_features = sample_features.sample(n=bagsize)
        sel_features.append(sample_features)
    sel_features = pd.concat(sel_features, axis=0)
else:
    sel_features = all_features.sort_values(order_by_col).groupby('ID').head(bagsize).sort_values(['ID', order_by_col])
    sel_features = sel_features.drop(order_by_col, axis=1)
sel_features

In [ ]:
import pandas as pd
from onekey_algo import get_param_in_cwd
import numpy as np
import os

label_data = pd.read_csv(get_param_in_cwd('label_file'), dtype={'ID': str})
data_root = get_param_in_cwd('data_root')
os.makedirs(data_root, exist_ok=True)
for ug in get_param_in_cwd('subsets'):
    sub_group = label_data[label_data['group'] == ug]
    display(sub_group)
    sub_features = pd.merge(sel_features, sub_group[['ID']], on='ID', how='inner')
    print(ug, len(np.unique(sub_group['ID'])), len(np.unique(sub_features['ID'])))
    sub_features.to_csv(f'{data_root}/{ug}.csv', index=False)

In [ ]:
import os
from onekey_algo import get_param_in_cwd
from onekey_algo.fusion.MultiTransformer.run_model import train_categorical_model as clf_main
from collections import namedtuple

data_root = get_param_in_cwd('data_root')
# 单中心设置参数
train = os.path.join(data_root, 'train.csv')
if 'val' in get_param_in_cwd('subsets'):
    val = os.path.join(data_root, 'val.csv')
    tests = [os.path.join(data_root, f'{subset}.csv') for subset in get_param_in_cwd('subsets')]
else:
    val = os.path.join(data_root, 'test.csv')
    tests = []

# 多中心设置参数
# train = os.path.join(data_root, 'train.csv')
# val = os.path.join(data_root, 'val.csv')
# tests = [os.path.join(data_root, subset + '.csv') for subset in get_param_in_cwd('subsets')
#          if subset not in ['train', 'val'] and os.path.exists(os.path.join(data_root, subset + '.csv'))]
print(tests)
# tests = [os.path.join(data_root, 'NHH-GC.csv')]
target_file = get_param_in_cwd('label_file')
input_dim = sel_features.shape[1] - 1
normalize = True
header = 0

bags_size = get_param_in_cwd('bagsize')
for bags_size in [bags_size]:
    params = dict(train=train,
                  val=val,
                  tests=tests,
                  target_file=target_file,
                  j=0,
                  input_dim=input_dim,
                  bags_size=bags_size,
                  normalize=normalize,
                  header=header,
                  gpus=[0],
                  batch_size=8,
                  model_name='Transformer',
                  epochs=get_param_in_cwd('transformer_epoch', 100),
                  init_lr=0.001,
                  optimizer='sgd',
                  model_root=os.path.join(get_param_in_cwd('model_root'), f'Transformer'),
                  add_date=False,
                  retrain='',
                  iters_start=0,
                  iters_verbose=1,
                  save_per_epoch=True,
                  pretrained=True)
    # 训练模型
    Args = namedtuple("Args", params)
    clf_main(Args(**params))

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from onekey_algo.custom.components import metrics
from onekey_algo.custom.components.comp1 import draw_roc
from onekey_algo import get_param_in_cwd
import numpy as np

def get_log(epoch):
    log_ = pd.concat([pd.read_csv(os.path.join(root, f'../train/Epoch-{epoch}_spec.csv'), dtype={'ID': str}), 
                      pd.read_csv(os.path.join(root, f'../valid/Epoch-{epoch}_spec.csv'), dtype={'ID': str})], axis=0)
    log_.columns = ['ID', 'label-0', 'label-1']
    log_['ID'] = log_['ID'].astype(str)
    return log_

os.makedirs('img', exist_ok=True)
label_data = pd.read_csv(get_param_in_cwd('label_file'), dtype={'ID': str})
ep_map = {bagsize: get_param_in_cwd('sel_epoch')['transformer']}
metrics_df = []
dim = 1024
for dim in ep_map.keys():
    model_root = os.path.join(get_param_in_cwd('model_root'), f'Transformer')
    root = os.path.join(model_root, 'Transformer', 'viz')
    all_gt = []
    all_pred = []
    metric_df = []
    for idx, subset in enumerate(get_param_in_cwd('subsets')):
        sub_group = get_log(ep_map[dim])
        sub_group.columns = ['ID', 'label-0', 'label-1']
        sub_group = pd.merge(sub_group, label_data[label_data['group'] == subset], on='ID', how='inner')
        sub_group[['ID', 'label-0', 'label-1']].to_csv(f'results/Transformer_Transformer_{subset}.csv', index=False)
        all_gt.append(np.array(sub_group['label']))
        all_pred.append(np.array(sub_group['label-1']))
        acc, auc, ci, tpr, tnr, ppv, npv, _, _, _, thres = metrics.analysis_pred_binary(np.array(sub_group['label']), 
                                                                                        np.array(sub_group['label-1']))
        ci = f"{ci[0]:.4f}-{ci[1]:.4f}"
        metric_df.append([acc, auc, ci, tpr, tnr, ppv, npv, thres, subset])
    metric_df = pd.DataFrame(metric_df, columns=['Acc', 'AUC', '95% CI', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Youden', 'Cohort'])
    metric_df['Dim'] = dim
    metrics_df.append(metric_df)
    draw_roc(all_gt, all_pred, labels=get_param_in_cwd('subsets'), title=f"Transformer {dim} Dimension CNN Features")
    plt.savefig(f'img/Transformer_Transformer_roc.svg', bbox_inches='tight')
    plt.show()
metrics_df = pd.concat(metrics_df, axis=0)
metric_df.to_csv('results/Transformer_cmp.csv', index=False)
metrics_df